## python project 

In [10]:
# import our library
import pandas as pd
from pathlib import Path
from datetime import datetime

In [11]:
# Project paths
BASE_DIR = Path.cwd()
RAW_FOLDER = BASE_DIR/"data"/"raw"
CLEAN_FOLDER = BASE_DIR/"data"/"cleaned"
COMBINE_FOLDER = BASE_DIR/"data"/"combined"
LOG_FOLDER = BASE_DIR/"data"/"logs"

# create folder if they don't exists
RAW_FOLDER.mkdir(parents=True, exist_ok=True)
CLEAN_FOLDER.mkdir(parents=True, exist_ok=True)
COMBINE_FOLDER.mkdir(parents=True, exist_ok=True)
LOG_FOLDER.mkdir(parents=True, exist_ok=True)

MASTER_FILE = COMBINE_FOLDER/"master_data.csv"
PROCESSED_FILE = BASE_DIR/"processed_files.txt"


In [12]:
today = datetime.today()
today = today.strftime('%Y-%m-%d')

LOG_FILE = LOG_FOLDER/f'pipeline_{today}.log'

def log(message):
    time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    msg = f'[{time}] {message}'
    print(msg)

    with open(LOG_FILE, 'a', encoding='utf-8') as file:
        file.write(msg + '\n')

In [13]:
# load processed files
if PROCESSED_FILE.exists():
    with open(PROCESSED_FILE, 'r', encoding='utf-8') as file:
        processed_files = set(file.read().splitlines())
else: 
    processed_files = set()
processed_files

set()

In [14]:
# Read files
all_files = list(RAW_FOLDER.glob('*.csv'))

new_files = [file for file in all_files if file.name not in processed_files]

if not (new_files):
    log('No new files found')
    exit()

log(f'found {len(new_files)} new file(s)')
new_files

[2026-10-06 16:27:54] found 21 new file(s)


[WindowsPath('C:/Users/UCHE/OneDrive/Documents/python-Basics-for-Data-analytics/project/data/raw/warehouse_asaba_sales_2026-05-25.csv'),
 WindowsPath('C:/Users/UCHE/OneDrive/Documents/python-Basics-for-Data-analytics/project/data/raw/warehouse_asaba_sales_2026-05-26.csv'),
 WindowsPath('C:/Users/UCHE/OneDrive/Documents/python-Basics-for-Data-analytics/project/data/raw/warehouse_asaba_sales_2026-05-27.csv'),
 WindowsPath('C:/Users/UCHE/OneDrive/Documents/python-Basics-for-Data-analytics/project/data/raw/warehouse_asaba_sales_2026-05-28.csv'),
 WindowsPath('C:/Users/UCHE/OneDrive/Documents/python-Basics-for-Data-analytics/project/data/raw/warehouse_asaba_sales_2026-05-29.csv'),
 WindowsPath('C:/Users/UCHE/OneDrive/Documents/python-Basics-for-Data-analytics/project/data/raw/warehouse_asaba_sales_2026-05-30.csv'),
 WindowsPath('C:/Users/UCHE/OneDrive/Documents/python-Basics-for-Data-analytics/project/data/raw/warehouse_asaba_sales_2026-05-31.csv'),
 WindowsPath('C:/Users/UCHE/OneDrive/Docu

In [15]:
# cleaning our data 
cleaned_data = []
for file in new_files:
    try:
        # create table
        data = pd.read_csv(file)

        # standardize column name
        data.columns = data.columns.str.lower().str.strip().str.replace(" ", "_")
    
        # Handling missing values
        before = len(data)
        data = data.dropna()
        after = len(data)
        log(f"{file.name}: " + f" removed {before - after} rows")
    
        # change the text in warehouse_region to titlecase, sku_code to uppercase , product_name and category title
        data.loc[:,'warehouse_region'] =  data['warehouse_region'].str.title()
        data.loc[:,'sku_code'] =  data['sku_code'].str.upper()
        data.loc[:,'product_name'] =  data['product_name'].str.title()
        data.loc[:,'category'] =  data['category'].str.title()
    
        # fix the issue with sku_code
        def fix_sku(d):
            s1 = d[0:3]
            s2 = d[-3:]
            answer = s1 + '-' + s2
            return answer
        
        data.loc[:,'sku_code'] = data['sku_code'].apply(func= fix_sku)
    
        # fix the date
        # data.loc[:,'date'] = pd.to_datetime(arg = data['date'], errors = 'raise', dayfirst= True, format='mixed')
        data.loc[:,'date_fixed'] = pd.to_datetime(arg = data['date'], errors = 'coerce', dayfirst= True, format='mixed')
        data.drop(columns =['date'], inplace=True)
        data.rename(columns = {'date_fixed' : 'date'} , inplace = True)
        
    
        cleaned_data.append(data)
    except Exception as e:
        log(f'ERROR processing {file.name}: {e}')

[2026-10-06 16:27:58] warehouse_asaba_sales_2026-05-25.csv:  removed 1 rows
[2026-10-06 16:27:58] warehouse_asaba_sales_2026-05-26.csv:  removed 5 rows
[2026-10-06 16:27:58] warehouse_asaba_sales_2026-05-27.csv:  removed 1 rows
[2026-10-06 16:27:58] warehouse_asaba_sales_2026-05-28.csv:  removed 1 rows
[2026-10-06 16:27:58] warehouse_asaba_sales_2026-05-29.csv:  removed 1 rows
[2026-10-06 16:27:58] warehouse_asaba_sales_2026-05-30.csv:  removed 3 rows
[2026-10-06 16:27:58] warehouse_asaba_sales_2026-05-31.csv:  removed 1 rows
[2026-10-06 16:27:58] warehouse_benin_sales_2026-05-25.csv:  removed 1 rows
[2026-10-06 16:27:58] warehouse_benin_sales_2026-05-26.csv:  removed 1 rows
[2026-10-06 16:27:59] warehouse_benin_sales_2026-05-27.csv:  removed 2 rows
[2026-10-06 16:27:59] warehouse_benin_sales_2026-05-28.csv:  removed 2 rows
[2026-10-06 16:27:59] warehouse_benin_sales_2026-05-29.csv:  removed 1 rows
[2026-10-06 16:27:59] warehouse_benin_sales_2026-05-30.csv:  removed 0 rows
[2026-10-06 

In [16]:
# combine all data 
new_data = pd.concat(cleaned_data, ignore_index=True)
new_data

,warehouse_region,sku_code,product_name,category,qty_sold,unit_price,closing_stock,date
0,Asaba,SKU-005,Peak Milk Tin 400G,Beverages,20.0,1292.48,44.0,2026-05-25
1,Asaba,SKU-015,Close-Up Toothpaste 140G,Personal Care,22.0,856.06,287.0,2026-05-25
2,Asaba,SKU-013,Dettol Antiseptic 500Ml,Household,20.0,1961.77,111.0,2026-05-25
3,Asaba,SKU-007,Bigi Cola 50Cl (Pack Of 12),Beverages,24.0,3879.54,178.0,2026-05-25
4,Asaba,SKU-002,Dangote Sugar 1Kg,Packaged Foods,18.0,943.80,88.0,2026-05-25
...,...,...,...,...,...,...,...,...
313,Onitsha,SKU-017,Always Ultra Pads (Pack Of 8),Personal Care,20.0,1397.76,156.0,2026-05-31
314,Onitsha,SKU-012,Sunlight Dishwashing Liquid 900Ml,Household,26.0,1615.41,94.0,2026-05-31
315,Onitsha,SKU-013,Dettol Antiseptic 500Ml,Household,25.0,1902.23,366.0,2026-05-31
316,Onitsha,SKU-011,Omo Detergent 900G,Household,18.0,2529.44,246.0,2026-05-31


In [8]:
# Save clean data


daily_file =  CLEAN_FOLDER/f'cleaned_data_{today}.csv'

new_data.to_csv(daily_file, index=False)

log(f'Cleaned data has been Saved')


[2026-08-20 20:36:25] Cleaned data has been Saved


In [18]:
# create and save master file

if MASTER_FILE.exists():
    master_data = pd.read_csv(MASTER_FILE)
    log(f'Existing master records: {len(master_data)}')
else: 
    master_data = pd.DataFrame()
    log('Master dataset does not exists. Creating it')

# combine master data to the newly cleaned data
if not master_data.empty:
    master_data = pd.concat([master_data, new_data], ignore_index=True)

else:
    master_data = new_data

# remove duplicate from master data
before = len(master_data)
master_data = master_data.drop_duplicates()
after = len(master_data)
log(f'we removed {before - after} duplicate rows')

# save master data to combined folder
master_data.to_csv(MASTER_FILE , index=False)
log(f'Master dataset saved')

[2026-10-06 16:42:55] Existing master records: 309
[2026-10-06 16:42:55] we removed 9 duplicate rows
[2026-10-06 16:42:55] Master dataset saved


In [10]:
# mark files as processed
with open(PROCESSED_FILE, 'a', encoding='utf-8') as file:
    for processed in new_files:
        file.write(processed.name + '\n')
log(f'Clean pipe completed successfully')
log(f'Total master records:  {len(master_data)}')

[2026-08-20 20:36:28] Clean pipe completed successfully
[2026-08-20 20:36:28] Total master records:  309
